## Download Manifesto Project Data

This notebook downloads data from the manifesto project through the download path in ```project/src/data/download_manifesto.py```.

It uses Japan, finds a manifesto row that has machine-readable text, and downloads one original-language manifesto preview.

In [57]:
from pathlib import Path
import getpass
import os
import sys
import pandas as pd

In [58]:

from src.data.download_manifesto import DownloadManifesto, download_country_manifestos
api_key = os.getenv("MANIFESTO_API") or getpass.getpass("Manifesto API key: ")
downloader = DownloadManifesto(dataset_key="MPDS2024a", version="2024-1", api_key=api_key)

## 1. Pull Japan rows from core datset

In [59]:
country_data = downloader.get_country_data("Japan")
assert country_data is not None and not country_data.empty, "No Japan rows returned from get_country_data()."
print(f"Japan rows in core dataset: {len(country_data)}")
country_data.head()

Japan rows in core dataset: 130


,countryname,party,partyname,date,keys
2948,Japan,71220,Japanese Communist Party,196011,71220_196011
2949,Japan,71320,Japan Socialist Party,196011,71320_196011
2950,Japan,71321,Democratic Socialist Party,196011,71321_196011
2951,Japan,71620,Liberal Democratic Party,196011,71620_196011
2952,Japan,71220,Japanese Communist Party,196311,71220_196311


## 2. Add corpus metadata and find text-available rows

Some rows in the Manifesto Project have only a PDF or metadata. This filters to only include rows with a ```manifesto_id``` (not na) before requesting text.

In [60]:
country_metadata, raw_metadata = downloader.get_metadata(country_data)
text_candidates = country_metadata[country_metadata["manifesto_id"].notna()].copy()
assert not text_candidates.empty, "No Japan rows have machine-readable Manifesto text metadata."

manifesto_count = len(text_candidates)
print(f"Rows with machine-readable manifesto_id: {manifesto_count}")
text_candidates[["countryname", "party", "partyname", "date", "keys", "manifesto_id", "language", "translation_en"]].head(10)

Rows with machine-readable manifesto_id: 9


,countryname,party,partyname,date,keys,manifesto_id,language,translation_en
116,Japan,71220,Japanese Communist Party,201412,71220_201412,71220_201412,japanese,False
117,Japan,71320,Social Democratic Party,201412,71320_201412,71320_201412,japanese,False
123,Japan,71220,Japanese Communist Party,201710,71220_201710,71220_201710,japanese,False
124,Japan,71320,Social Democratic Party,201710,71320_201710,71320_201710,japanese,False
125,Japan,71430,Japan Restoration Party,201710,71430_201710,71430_201710,japanese,False
126,Japan,71440,Constitutional Democratic Party of Japan,201710,71440_201710,71440_201710,japanese,False
127,Japan,71530,New Clean Government Party,201710,71530_201710,71530_201710,japanese,False
128,Japan,71620,Liberal Democratic Party,201710,71620_201710,71620_201710,japanese,False
129,Japan,71627,Party of Hope,201710,71627_201710,71627_201710,japanese,False


## 3. Download Manifesto Texts

In [61]:
one_row = text_candidates.head(manifesto_count)
one_manifesto = downloader.get_texts(one_row)
preview = one_manifesto[["party", "partyname", "manifesto_id", "text"]].copy()
preview

party                                 partyname  \
countryname date                                                      
Japan       201412  71220                  Japanese Communist Party   
            201412  71320                   Social Democratic Party   
            201710  71220                  Japanese Communist Party   
            201710  71320                   Social Democratic Party   
            201710  71430                   Japan Restoration Party   
            201710  71440  Constitutional Democratic Party of Japan   
            201710  71530                New Clean Government Party   
            201710  71620                  Liberal Democratic Party   
            201710  71627                             Party of Hope   

                    manifesto_id  \
countryname date                   
Japan       201412  71220_201412   
            201412  71320_201412   
            201710  71220_201710   
            201710  71320_201710   
            201710  71430_201710   
            201710  71440_201710   
            201710  71530_201710   
            201710  71620_201710   
            201710  71627_201710   

                                                                 text  
countryname date                                                       
Japan       201412  安倍政権の暴走ストップ！ 国民の声が生きる新しい政治を 日本共産党の総選挙政策 日本共産党 ...  
            201412  約束１　アベノミクスによる生活破壊を許さず、拡大した格差を是正します （１）景気を悪化させる...  
            201710  日本共産党の総選挙政策 日本共産党 安倍首相は、臨時国会の冒頭解散に打って出ました。 「森友...  
            201710  くらし支えます 1家計を温めボトムアップの経済政策でくらしの再建 憲法１３条の幸福追求権、２...  
            201710  消費増税凍結! 維新ならできる! 増税なしで改革実現! 身を切る改革で財源を生み出し、議員報...  
            201710  1 生活の現場から暮らしを立て直します アベノミクスの成果は上がらず、国民の所得を削り、中間...  
            201710  教育負担の軽減へ。 衆院選で公明党は、「教育負担の軽減へ。」を掲げます。 国づくりの基...  
            201710  北朝鮮の脅威から、 国民を守り抜きます わが国の上空を飛び越える弾道ミサイルの相次ぐ発...  
            201710  私たちが希求するものは、党の利益ではなく、議員の利益でもなく、 国民のため、つまり国民...

## 4. Sort Manifestos

Index by date, convert to year-month date-time format, sort manifestos by date, and drop country name and party id.

In [62]:
manifestos_indexed = preview.reset_index(drop=False)
manifestos_indexed['date'] = pd.to_datetime(manifestos_indexed['date'], format='%Y%m').dt.to_period('M')
manifestos_indexed = manifestos_indexed.set_index('date')
manifestos_indexed = manifestos_indexed.drop(columns=['countryname', 'party'])
manifestos_sorted = manifestos_indexed.sort_index()
manifestos_sorted

,partyname,manifesto_id,text
date,,,
2014-12,Japanese Communist Party,71220_201412,安倍政権の暴走ストップ！ 国民の声が生きる新しい政治を 日本共産党の総選挙政策 日本共産党 ...
2014-12,Social Democratic Party,71320_201412,約束１ アベノミクスによる生活破壊を許さず、拡大した格差を是正します （１）景気を悪化させる...
2017-10,Japanese Communist Party,71220_201710,日本共産党の総選挙政策 日本共産党 安倍首相は、臨時国会の冒頭解散に打って出ました。 「森友...
2017-10,Social Democratic Party,71320_201710,くらし支えます 1家計を温めボトムアップの経済政策でくらしの再建 憲法１３条の幸福追求権、２...
2017-10,Japan Restoration Party,71430_201710,消費増税凍結! 維新ならできる! 増税なしで改革実現! 身を切る改革で財源を生み出し、議員報...
2017-10,Constitutional Democratic Party of Japan,71440_201710,1 生活の現場から暮らしを立て直します アベノミクスの成果は上がらず、国民の所得を削り、中間...
2017-10,New Clean Government Party,71530_201710,教育負担の軽減へ。 衆院選で公明党は、「教育負担の軽減へ。」を掲げます。 国づくりの基...
2017-10,Liberal Democratic Party,71620_201710,北朝鮮の脅威から、 国民を守り抜きます わが国の上空を飛び越える弾道ミサイルの相次ぐ発...
2017-10,Party of Hope,71627_201710,私たちが希求するものは、党の利益ではなく、議員の利益でもなく、 国民のため、つまり国民...


## 5. Save dataframe to csv

In [63]:
manifestos_sorted.to_csv('Japan_sorted_manifestos.csv')